# 06 — Service Layer: activity plan suggestions

Ties everything together: `find_pois` (category/amenity matching, from KG
Modelling) + `GtfsRouter` (travel time, from the Reasoning Layer) into
`service/activity_planner.py`'s `plan_activities()` — which generates actual
multi-stop **plans**, not just ranked POI lists. Matches the one-pager's
motivation directly ("someone else planned a day... for you") rather than
just "find the nearest X."

Scope, stated plainly (see `service/activity_planner.py`'s docstring):
- Plans are at most 2 stops (origin → stop1 → stop2).
- The time budget is a **travel-time** budget only — it does not account for
  time actually spent at each stop (visiting a museum, walking a dog).

Two ways to use this notebook: a couple of fixed, reproducible example
queries (Sections 1-2) for reliability, and a live interactive widget
(Section 4) for typing in your own query — the widget cell won't show
meaningful output when run headlessly via `nbconvert`/export, it needs an
actual running Jupyter kernel with a click.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from rdflib import Graph
from reasoning.gtfs_routing import GtfsRouter
from service.activity_planner import plan_activities, format_plan
import matplotlib.pyplot as plt

g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
router = GtfsRouter(date="20260815")
print(f"KG: {len(g)} triples, GTFS router ready")

# a few well-connected, recognizable named origins -- avoids ever having to
# type raw lon/lat (which is exactly how an earlier test session found a real
# bug: (lon, lat) typed in the wrong order looks like valid coordinates but
# silently points somewhere off the map)
LANDMARKS = {
    "Karlsplatz": (16.368948, 48.200955),
    "Stephansplatz": (16.372931, 48.208611),
    "Praterstern": (16.393889, 48.216111),
    "Schwedenplatz": (16.377500, 48.212222),
    "Westbahnhof": (16.337778, 48.196389),
}

## 1. Single-stop example: a dog-friendly park, 30-minute travel budget

In [ ]:
origin_lon, origin_lat = LANDMARKS["Karlsplatz"]

plans = plan_activities(g, router, origin_lon, origin_lat,
                         interests=[{"label": "a dog-friendly park", "poi_classes": ["Park"],
                                     "required_amenities": ["Dogs allowed"]}],
                         time_budget_min=30, depart_after="14:00:00")
print(f"{len(plans)} option(s):\n")
for p in plans[:3]:
    print(format_plan(p, "14:00:00"))
    print()

## 2. A real 2-stop plan: dog-friendly park, then a library

In [ ]:
plans2 = plan_activities(g, router, origin_lon, origin_lat,
                          interests=[
                              {"label": "a dog-friendly park", "poi_classes": ["Park"], "required_amenities": ["Dogs allowed"]},
                              {"label": "a library", "poi_classes": ["Library"]},
                          ],
                          time_budget_min=45, depart_after="14:00:00")
print(f"{len(plans2)} plan(s):\n")
for p in plans2[:3]:
    print(format_plan(p, "14:00:00"))
    print()

best_plan = plans2[0]

## 3. What the best plan actually looks like on the map

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

stop_lons = router.stops["stop_lon"].to_numpy()
stop_lats = router.stops["stop_lat"].to_numpy()
ax.scatter(stop_lons, stop_lats, s=3, color="lightgray", zorder=1)

points = [(origin_lon, origin_lat, "origin (Karlsplatz)")]
for stop in best_plan["stops"]:
    points.append((stop["lon"], stop["lat"], f"{stop['label']}: {stop['name']}"))

xs, ys = zip(*[(p[0], p[1]) for p in points])
ax.plot(xs, ys, "--", color="#4C72B0", zorder=2)
for lon, lat, label in points:
    ax.scatter([lon], [lat], s=200, zorder=3)
    ax.annotate(label, (lon, lat), textcoords="offset points", xytext=(8, 8), fontsize=9)

# auto-zoom to the plan itself (with padding) instead of the whole city --
# a short local plan is otherwise an invisible dot on a city-wide map
pad = max(0.01, (max(xs) - min(xs)) * 0.6, (max(ys) - min(ys)) * 0.6)
ax.set_xlim(min(xs) - pad, max(xs) + pad)
ax.set_ylim(min(ys) - pad, max(ys) + pad)

ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"Suggested plan -- {best_plan['total_travel_min']:.0f} min total travel")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. Interactive: ask for your own plan

Pick a starting landmark, one or two interests, an optional required amenity
for each, and a travel-time budget. Leaving interest 2 as "(none)" gives a
single-stop suggestion instead of a 2-stop plan.

**Needs a live kernel to actually use** — running this notebook via
`nbconvert --execute` (or any headless export) renders the widgets but can't
simulate a button click, so this cell alone won't show a result outside a
real Jupyter session. Sections 1-2 above already demonstrate the exact same
underlying call, so the logic is validated either way.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

POI_OPTIONS = ["Museum", "Library", "BathingSite", "Park", "SwimmingPool", "PlaygroundArea", "TouristAttraction"]

origin_dd = widgets.Dropdown(options=list(LANDMARKS.keys()), description="Start at:")
interest1_dd = widgets.Dropdown(options=POI_OPTIONS, value="Park", description="Interest 1:")
amenity1_txt = widgets.Text(description="Amenity 1:", placeholder="e.g. Dogs allowed (optional)")
interest2_dd = widgets.Dropdown(options=["(none)"] + POI_OPTIONS, value="Library", description="Interest 2:")
amenity2_txt = widgets.Text(description="Amenity 2:", placeholder="optional")
budget_slider = widgets.IntSlider(value=45, min=10, max=180, step=5, description="Budget (min):")
depart_txt = widgets.Text(value="14:00:00", description="Depart after:")
go_button = widgets.Button(description="Find a plan", button_style="primary")
output = widgets.Output()

def on_click(_):
    with output:
        clear_output()
        lon, lat = LANDMARKS[origin_dd.value]
        interests = [{"label": interest1_dd.value, "poi_classes": [interest1_dd.value],
                      "required_amenities": [amenity1_txt.value] if amenity1_txt.value else None}]
        if interest2_dd.value != "(none)":
            interests.append({"label": interest2_dd.value, "poi_classes": [interest2_dd.value],
                               "required_amenities": [amenity2_txt.value] if amenity2_txt.value else None})
        plans = plan_activities(g, router, lon, lat, interests=interests,
                                 time_budget_min=budget_slider.value, depart_after=depart_txt.value)
        if not plans:
            print("No plan found within that budget -- try a larger budget or a different combination.")
        for p in plans[:3]:
            print(format_plan(p, depart_txt.value))
            print()

go_button.on_click(on_click)

display(widgets.VBox([origin_dd, interest1_dd, amenity1_txt, interest2_dd, amenity2_txt,
                       budget_slider, depart_txt, go_button, output]))

## Findings

**Multi-stop plans work, and the chaining is correct** — the arrival clock
time at stop 1 becomes the departure time for the search to stop 2 (not just
"budget minus travel minutes"), so leg 2 genuinely reflects what's reachable
*from stop 1 at that specific time*, matching real schedule constraints
rather than treating the trip as two independent lookups.

**Performance is acceptable for interactive use but not instant** — a
2-interest plan takes a few seconds (each of the top stop-1 candidates needs
its own `reachable_from()` call for the second leg), noticeably slower than
the sub-second single-stop lookups from earlier notebooks. Fine for a
click-and-wait demo, would need caching or a smaller candidate pool for
anything more real-time.

**What's still not here:** live data (RBL/disruptions — a plan could
currently suggest a route that's actually disrupted right now) and any
modelling of time spent at each stop. Both are natural next steps, not
oversights — see `docs/reasoning_layer_decisions.md`.